In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
PROJECT_PATH = "/content/drive/MyDrive/Serie_A_Forecast"

In [3]:
# =========================
# 03 MATCH PREDICTIONS
# Serie A Match Prediction
# =========================

import pandas as pd
import numpy as np
import joblib

# =========================
# LOAD DATA AND MODEL
# =========================

df_final = pd.read_csv(
    f"{PROJECT_PATH}/data/processed/df_final.csv"
)

#df_final = pd.read_csv("df_final.csv")

model = joblib.load(
    f"{PROJECT_PATH}/models/logistic_model.pkl"
)

#model = joblib.load("logistic_model.pkl")

df_final.head()

,Date,HomeTeam,AwayTeam,FTR,home_avg_goals_5,away_avg_goals_5,home_avg_goals_conceded_5,away_avg_goals_conceded_5,home_avg_points_5,away_avg_points_5,home_avg_shots_5,away_avg_shots_5,home_avg_shots_target_5,away_avg_shots_target_5,home_home_points_5,away_away_points_5,home_home_goals_5,away_away_goals_5
0,2025-09-13,Cagliari,Parma,H,0.5,0.5,1.0,1.5,0.5,0.5,12.0,7.5,3.5,2.5,1.0,0.0,1.0,0.0
1,2025-09-14,Roma,Torino,A,1.0,0.0,0.0,2.5,3.0,0.5,12.0,9.5,4.5,4.0,3.0,0.0,1.0,0.0
2,2025-09-14,Atalanta,Lecce,H,1.0,0.0,1.0,1.0,1.0,0.5,16.0,7.0,4.5,1.0,1.0,1.0,1.0,0.0
3,2025-09-14,Pisa,Udinese,A,0.5,1.5,1.0,1.0,0.5,2.0,9.0,12.0,2.5,4.5,0.0,3.0,0.0,2.0
4,2025-09-14,Sassuolo,Lazio,H,1.0,2.0,2.5,1.0,0.0,1.5,8.0,13.5,4.0,3.5,0.0,0.0,0.0,0.0


In [4]:
# =========================
# PREDICTION FUNCTION
# Uses last 5 HOME matches for home team
# and last 5 AWAY matches for away team
# =========================

def predict_match(df_final, home_team, away_team, model):

    home_prev = df_final[
        df_final["HomeTeam"] == home_team
    ].tail(5)

    away_prev = df_final[
        df_final["AwayTeam"] == away_team
    ].tail(5)

    if len(home_prev) == 0:
        raise ValueError(f"No home matches found for {home_team}")

    if len(away_prev) == 0:
        raise ValueError(f"No away matches found for {away_team}")

    new_match = pd.DataFrame({
        "home_avg_goals_5": [home_prev["home_avg_goals_5"].mean()],
        "away_avg_goals_5": [away_prev["away_avg_goals_5"].mean()],

        "home_avg_goals_conceded_5": [home_prev["home_avg_goals_conceded_5"].mean()],
        "away_avg_goals_conceded_5": [away_prev["away_avg_goals_conceded_5"].mean()],

        "home_avg_points_5": [home_prev["home_avg_points_5"].mean()],
        "away_avg_points_5": [away_prev["away_avg_points_5"].mean()],

        "home_avg_shots_5": [home_prev["home_avg_shots_5"].mean()],
        "away_avg_shots_5": [away_prev["away_avg_shots_5"].mean()],

        "home_avg_shots_target_5": [home_prev["home_avg_shots_target_5"].mean()],
        "away_avg_shots_target_5": [away_prev["away_avg_shots_target_5"].mean()],

        "home_home_points_5": [home_prev["home_home_points_5"].mean()],
        "away_away_points_5": [away_prev["away_away_points_5"].mean()],

        "home_home_goals_5": [home_prev["home_home_goals_5"].mean()],
        "away_away_goals_5": [away_prev["away_away_goals_5"].mean()]
    })

    prediction = model.predict(new_match)[0]
    probabilities = model.predict_proba(new_match)[0]

    result = pd.DataFrame({
        "Result": model.classes_,
        "Probability": probabilities
    })

    result["Probability"] = result["Probability"].round(4)

    result = result.sort_values(
        by="Probability",
        ascending=False
    ).reset_index(drop=True)

    return prediction, result, new_match

In [5]:
# =========================
# EXAMPLE: LECCE VS JUVENTUS
# =========================

prediction, probabilities, features_used = predict_match(
    df_final=df_final,
    home_team="Lecce",
    away_team="Juventus",
    model=model
)

print("Predicted result:", prediction)

probabilities

Predicted result: A


,Result,Probability
0,A,0.6899
1,D,0.2155
2,H,0.0946


In [6]:
# =========================
# FEATURES USED BY THE MODEL
# =========================

features_used

,home_avg_goals_5,away_avg_goals_5,home_avg_goals_conceded_5,away_avg_goals_conceded_5,home_avg_points_5,away_avg_points_5,home_avg_shots_5,away_avg_shots_5,home_avg_shots_target_5,away_avg_shots_target_5,home_home_points_5,away_away_points_5,home_home_goals_5,away_away_goals_5
0,0.68,2.16,1.32,1.08,0.8,1.84,9.96,17.92,2.04,6.48,1.04,1.8,0.68,2.16


In [7]:
# =========================
# MAKE RESULT MORE READABLE
# =========================

def explain_prediction(prediction):

    if prediction == "H":
        return "Home win"
    elif prediction == "D":
        return "Draw"
    elif prediction == "A":
        return "Away win"
    else:
        return "Unknown"


print("Prediction:", explain_prediction(prediction))

probabilities["Result_Label"] = probabilities["Result"].map({
    "H": "Home win",
    "D": "Draw",
    "A": "Away win"
})

probabilities

Prediction: Away win


,Result,Probability,Result_Label
0,A,0.6899,Away win
1,D,0.2155,Draw
2,H,0.0946,Home win


In [8]:
# =========================
# SAVE SINGLE MATCH PREDICTION
# =========================


probabilities.to_csv(
    f"{PROJECT_PATH}/outputs/lecce_juventus_prediction.csv",
    index=False
)

#probabilities.to_csv(
#    "lecce_juventus_prediction.csv",
 #   index=False
#)

print("Prediction saved successfully.")

Prediction saved successfully.
